# Loading files and helper functions

In [1]:
import numpy as np
import pickle
import torch
import numpy as np
from torch.utils.data import Dataset


# TODO: add the right paths
cleaned_adherence_data = np.load("smoking_data_for_ML.npy")
with open("list_persuasiveMessages_smoking.pkl", "rb") as f:
    list_persuasiveMessages_smoking = pickle.load(f)
with open("list_activities_smoking.pkl", "rb") as f:
    list_activities_smoking = pickle.load(f)


def create_feature_mean_dict(train_array):
    # Flatten the first two dimensions (people and sessions) 
    # so we can treat every observation as an independent sample
    # New shape: (n_people * n_sessions, n_dimensions)
    flat_data = train_array.reshape(-1, train_array.shape[-1])
    
    # Extract the relevant dimensions
    # dim_0 = values to average, dim_1 = key 'a', dim_2 = key 'b'
    vals = flat_data[:, 0]
    dim_1 = flat_data[:, 1]
    dim_2 = flat_data[:, 2]
    
    # Combine dim_1 and dim_2 into a single array of pairs
    # This makes it easier to find unique combinations
    keys = np.column_stack((dim_1, dim_2))
    
    # Find unique rows (pairs) and their indices
    unique_keys, indices = np.unique(keys, axis=0, return_inverse=True)
    
    # Calculate means for each unique pair
    # We use np.bincount for speed; it sums 'vals' grouped by 'indices'
    sums = np.bincount(indices, weights=vals)
    counts = np.bincount(indices)
    means = sums / counts
    
    # Zip the unique pairs with their means into the final dictionary
    # Convert keys to tuples so they are hashable
    result_dict = {tuple(key): mean for key, mean in zip(unique_keys, means)}
    
    return result_dict

class AdherenceDataset(Dataset):
    """
    Dataset for full trajectories.
    
    Returns (X, Y) where:
    - X: (T, D-1) control variables over time
    - Y: (T,) adherence target over time (as class indices)
    """
    
    def __init__(self, data, target_as_classes=True):
        """
        Parameters
        ----------
        data : np.ndarray
            Shape (N, T, D) where:
            - data[..., 0] is adherence (can be class indices or continuous values)
            - data[..., 1:] are control variables
        target_as_classes : bool
            If True, treats target as class indices (integers).
            If False, treats target as continuous values (floats).
            Default: True
        """
        self.X = torch.tensor(data[..., 1:], dtype=torch.float32)  # (N, T, D-1)
        
        # Convert target to appropriate type
        if target_as_classes:
            # Ensure target is integer (class indices)
            self.Y = torch.tensor(data[..., 0], dtype=torch.long)   # (N, T)
        else:
            # Keep target as float (continuous values)
            self.Y = torch.tensor(data[..., 0], dtype=torch.float32)   # (N, T)
        
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]



# LLM testing pipeline
ends with saving the LLM MSEs in `llm_mses_acrossRuns.pkl`

In [ ]:
import os
import numpy as np

# TODO: optional, pick a subset of the data if full dataset makes prompts too large
cleaned_adherence_data = cleaned_adherence_data[:,:,:]

np.set_printoptions(threshold=np.inf)

import numpy as np
from torch.utils.data import random_split
import torch
from sklearn.metrics import mean_squared_error
import numpy as np
import copy

torch.manual_seed(0) # torch seed for reproducibility

# TODO: pick runs
N_RUNS = 3
seeds = [torch.randint(0, 1000000, size=(1,)).item() for _ in range(N_RUNS)]
with open("/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_smoking/seeds.pkl", "wb") as f:
    pickle.dump(seeds, f)
    
TRAIN_SPLIT = 0.8
BATCH_SIZE_VALUE = 4

reinforce_data = cleaned_adherence_data
dataset = AdherenceDataset(reinforce_data, target_as_classes=True)
# dataset = cleaned_adherence_data

dataset_size = len(dataset)
train_size = int(TRAIN_SPLIT * dataset_size)
val_size = dataset_size - train_size


train_losses_acrossRuns= []
llm_mses_acrossRuns = []

val_sets_acrossRuns = []

N = reinforce_data.shape[0]
T = reinforce_data.shape[1]
D = reinforce_data.shape[2]

for run_idx in range(N_RUNS):
    print(f"========= Run {run_idx+1} ============")
    c = 0
    train_set, val_set = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seeds[run_idx])
        )

    # ------------------ LLM setup ------------------
    import pickle
    from tarfile import HeaderError
    import numpy as np

    # indexing lists to give a dictionary for the activity_users_in_train_set part
    indexed_list_persuasiveMessages_smoking = [f"({i}) {msg}" for i, msg in enumerate(list_persuasiveMessages_smoking)]
    indxed_list_activities_smoking = [f"({i+1}) {msg}" for i, msg in enumerate(list_activities_smoking)]

    # ------------------ LLM inference ------------------

    num_sessions = cleaned_adherence_data.shape[1]

    y_true_llm = []
    y_pred_llm = []

    # Create val_array of shape (n_persons, n_timepoints, n_dimensions) where dim 0 = targets
    to_insert_in = copy.deepcopy(val_set[:][0])
    to_insert = copy.deepcopy(val_set[:][1])
    val_array = np.insert(to_insert_in, 0, to_insert, axis=2)

    to_insert_in = copy.deepcopy(train_set[:][0])
    to_insert = copy.deepcopy(train_set[:][1])
    train_array = np.insert(to_insert_in, 0, to_insert, axis=2)

    for idx_person in range(len(val_array)):
        for idx_session in range(1, num_sessions): # skipping inferencefor day 1 to match NDE evaluation

            y_pred_llm_oneSession = []
            # Prompt
            past_sessions_data = []
            for s_idx in range(idx_session):
                # Construct the string for each individual past session
                # Customize the indexing below based on your actual data structure
                effort = int(val_array[idx_person, s_idx, 0]) 
                
                session_info = (
                    f"Session {s_idx + 1}:\n"
                    f"  The effort you put into this activity: {effort}\n"
                    f"  The persuasion message you received: {list_persuasiveMessages_smoking[int(val_array[idx_person, s_idx, 1])]}\n"
                    # index gets -1 because the activities indexes start from 1
                    f"  The activity you were recommended: {list_activities_smoking[int(val_array[idx_person, s_idx, 2]) - 1]}\n"
                    f"  Type of activity: { "smoking cessation" if int(val_array[idx_person, s_idx, 3]) == 1 else "increasing physical activity"}"
                )
                past_sessions_data.append(session_info)
            # 2. Join them together or leave empty if no sessions exist
            if past_sessions_data:
                past_behavior_header = "\nPast Behavior:\nThe following is your previous effort to perform the activities recommended, the persuasion message we gave you, the activity we recommended you and the type of activity we recommended you.\n\n"
                past_behavior_text = past_behavior_header + "\n\n".join(past_sessions_data)
            else:
                past_behavior_text = ""

            past_sessions_data2 = []
            for person_idx in range(len(train_array)):
                # Construct the string for each individual past session
                # Customize the indexing below based on your actual data structure
                
                efforts = train_array[person_idx, :, 0].tolist()
                demographics = train_array[person_idx, 0, 4:].tolist()
                combined = efforts + demographics
                session_info = (
                    " ".join(str(x) for x in combined)
                        )
                past_sessions_data2.append(session_info)

            dict_mean_adherence = create_feature_mean_dict(train_array)
            intervention_combination = (float(train_array[idx_person,idx_session,1]), float(train_array[idx_person,idx_session,2]))

            if intervention_combination in dict_mean_adherence:
                header_activity_users_in_train_set = f"""
                    In case it is useful to make your decision, this is the mean adherence value for the other participants with the same persuasive message and action recommendation is {dict_mean_adherence[intervention_combination]} and below is an array with the decisions made by other users given their demographics; The rows represent the users, the first 4 values in the row are their effort in the 4 sessions and the rest of the values are these demographics: TTM_Smoking, TTM_PA, NFC, Pers_Extraversion, Pers_Agreeableness, Pers_Conscientiousness,Pers_ES, Pers_OE, Involvement, age, Gender identity, PA_Identity, Quitting_Self_Identity, Quit_Before_24h.
                    """
            else:
                header_activity_users_in_train_set = f"""
                In case it is useful to make your decision, below is an array with the decisions made by other users given their demographics; The rows represent the users, the first 4 values in the row are their effort in the 4 sessions and the rest of the values are these demographics: TTM_Smoking, TTM_PA, NFC, Pers_Extraversion, Pers_Agreeableness, Pers_Conscientiousness,Pers_ES, Pers_OE, Involvement, age, Gender identity, PA_Identity, Quitting_Self_Identity, Quit_Before_24h.
                """

            activity_users_in_train_set = header_activity_users_in_train_set + "\n\n".join(past_sessions_data2) 

            # ------- Logic to determine age based on bin number -------
            bin_number = int(val_array[idx_person, idx_session, 13])
            age_intervals = ["18-20", "21-25", "26-30", "31-35", "36-40", "41-45", "46-50", "51-55", "56-60", "61-65", "66-70", "71-74"]
            if 0 <= bin_number < len(age_intervals):
                # Retrieve the string directly
                interval = age_intervals[bin_number]
                age_message = f"Your age is between {interval} years old."
            else:
                age_message = ""
            # ------- End of logic to determine age -------

            input_text = f"""
            You are a smoker enrolled in a study where at each session you receive persuasive messages and activities meant to help you quit smoking. 
            Through this program you receive at each session a persuasive message and an activity recommendation.   
            Below is your background and history with the program.

            Your Background:
            Between preparation and contemplation, your quitting smoking stage is {"preparation" if val_array[idx_person, idx_session, 5].item() == 1 else "contemplation"} 
            Your level of becoming physically active is {val_array[idx_person, idx_session, 5].item()} where  (1) denotes Maintenance, (2) Action, (3) Preparation, (4) Contemplation, and (5) Precontemplation.
            Your need for cognition index is {val_array[idx_person, idx_session, 6].item()} on a 1-5 scale.
            Your extraversion is {val_array[idx_person, idx_session, 7].item()} on a 1-7 scale.
            Your Agreeableness is {val_array[idx_person, idx_session, 8].item()} on a 1-7 scale.
            Your Conscientiousness is {val_array[idx_person, idx_session, 9].item()} on a 1-7 scale.
            Your Emotional stability is {val_array[idx_person, idx_session, 10].item()} on a 1-7 scale.
            Your Openness to experience is {val_array[idx_person, idx_session, 11].item()} on a 1-7 scale.
            Your activity involvement is {val_array[idx_person, idx_session, 12].item()} on a scale of -5 to 5.
            {age_message}
            Your gender identity is {val_array[idx_person, idx_session, 14].item()} where  1 = Male, 2 = Female, and 3 = Other.
            Your physical activity identity index is {val_array[idx_person, idx_session, 15].item()} on a 1-5 scale.
            Your quitter self-identity index is {val_array[idx_person, idx_session, 16].item()} on a 1-5 scale.
            { "You previously quit smoking for at least 24 hours." if val_array[idx_person, idx_session, 17].item() == 1 else "You haven't previously quit smoking for at least 24 hours."}
            {past_behavior_text}

            Based on this information, the context of the program the typical behavior of smokers, and the fact that in the session today you got a persuasion message the following persuasion message, activity recommended and type of activity, decide what your effort you'll put in the recommended activity by picking an integer from 0 to 10 where 0 means no effort and 10 means extremely strong effort.

            Key Consideration: The effort you put into the recommended activity after a previous session does not necessarily imply the effort at the next. The effort should depend on your specific circumstances that on the day of that session.

            In your last session your persuasive message was:
            {list_persuasiveMessages_smoking[int(val_array[idx_person, idx_session, 1])]}\n

            Your recommended activity was:
            {list_activities_smoking[int(val_array[idx_person, idx_session, 2]) - 1]}

            Your activity type was:
            { "smoking cessation" if val_array[idx_person, idx_session, 3].item() == 1 else "increasing physical activity"}

            Question:  A few weeks past since your last session, by picking an integer from 0 to 10 where 0 means no effort and 10 means extremely strong effort, what is the effort you put into performing the recommended activity?

            Please respond with your final decision by only picking one integer from the interval [0-10], no other text should be included.
            
            {activity_users_in_train_set}
            """

            # ------------------ LLM evaluation ------------------
            import anthropic

            # TODO: Add your API key
            client = anthropic.Anthropic(api_key='sk-ant-XXX')

            message = client.messages.create(
                model="claude-haiku-4-5",
                max_tokens=1000,
                messages=[
                    {
                        "role": "user",
                        "content": input_text,
                    }
                ],
            )
            content = message.content[0].text

            if c % 10 == 0:
                print(f"Inference number {c}")
            c = c+1
            # ------------------ End section to change ------------------
            print("true:", int(val_array[idx_person, idx_session, 0]))
            try:
                y_pred_llm.append(int(content)) 
                print("pred: ", content)
            except (ValueError, TypeError):         # in case the llm outputs also its thinking, we take the last integer in the output
                import re
                numbers = re.findall(r'\d+(?:\.\d+)?', content)
                y_pred_llm.append(numbers[-1])
                print("cleaned pred:", numbers[-1])
            
            y_true_llm.append(val_array[idx_person, idx_session, 0])

    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_smoking/pred_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(y_pred_llm, f) 
    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_smoking/true_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(y_true_llm, f)
    mse_llm_run = mean_squared_error(y_true_llm, y_pred_llm)
    llm_mses_acrossRuns.append(mse_llm_run)
    with open(f"/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_smoking/mse_run_{run_idx+1}.pkl", "wb") as f:
      pickle.dump(mse_llm_run, f) 



# Save LLM MSEs across runs
with open("/Users/andreilixandru/Downloads/NDEs-for-adherence-main/RUNS_smoking/llm_mses_acrossRuns.pkl", "wb") as f:
    pickle.dump(llm_mses_acrossRuns, f)
